# Fraud Detection - Detection des Transactions Frauduleuses
### ECE Paris - Data & AI B3 | Meriem
---
**Dataset** : Kaggle Credit Card Fraud Detection
- 284 807 transactions europeennes anonymisees
- 492 fraudes (0.17%)
- Features : V1-V28 (PCA), Time, Amount, Class

In [ ]:
# Installation des librairies (a faire une seule fois)
# !pip install -r ../requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek
import xgboost as xgb
import lightgbm as lgb
import joblib
import mlflow
import mlflow.sklearn
import os

os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)
os.makedirs('../models', exist_ok=True)

GOLD  = '#d4af37'
RED   = '#ff6b6b'
GREEN = '#4ade80'

plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor':   '#111111',
    'axes.edgecolor':   GOLD,
    'axes.labelcolor':  '#e8d5a0',
    'xtick.color':      '#e8d5a0',
    'ytick.color':      '#e8d5a0',
    'text.color':       '#e8d5a0',
    'grid.color':       '#222222',
    'legend.facecolor': '#111111',
    'legend.edgecolor': GOLD,
})

print('Imports OK')

## 1. Chargement des Donnees

In [ ]:
df = pd.read_csv('../data/creditcard.csv')

print(f'Shape : {df.shape}')
print(f'Fraudes : {df["Class"].sum():,} ({df["Class"].mean()*100:.4f}%)')
print(f'Valeurs manquantes : {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe()

## 2. Analyse Exploratoire des Donnees (EDA)

### 2.1 Distribution des Classes

In [ ]:
class_counts = df['Class'].value_counts()
print(f'Classe 0 (Normal) : {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)')
print(f'Classe 1 (Fraude) : {class_counts[1]:,} ({class_counts[1]/len(df)*100:.2f}%)')
print(f'Ratio : 1 fraude pour {class_counts[0]//class_counts[1]} transactions normales')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des Classes', color=GOLD, fontsize=14)

bars = axes[0].bar(['Normal (0)', 'Fraude (1)'], class_counts.values, color=[GOLD, RED], alpha=0.85, edgecolor='black')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, f'{val:,}', ha='center', color=GOLD, fontweight='bold')
axes[0].set_title('Nombre de transactions', color=GOLD)
axes[0].set_ylabel('Count')

axes[1].pie(class_counts.values, labels=[f'Normal\n{class_counts[0]:,}', f'Fraude\n{class_counts[1]:,}'],
            colors=[GOLD, RED], autopct='%1.2f%%', startangle=90,
            textprops={'color': 'white'}, explode=(0, 0.08))
axes[1].set_title('Proportion', color=GOLD)
plt.tight_layout()
plt.savefig('../outputs/figures/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Analyse des Montants

In [ ]:
fraud_amounts  = df[df['Class'] == 1]['Amount']
normal_amounts = df[df['Class'] == 0]['Amount']

print(f'Montant moyen Normal : {normal_amounts.mean():.2f} EUR')
print(f'Montant moyen Fraude : {fraud_amounts.mean():.2f} EUR')
print(f'Montant max   Fraude : {fraud_amounts.max():.2f} EUR')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse des Montants', color=GOLD, fontsize=14)

axes[0].hist(normal_amounts, bins=60, color=GOLD, alpha=0.65, label='Normal', density=True)
axes[0].hist(fraud_amounts,  bins=50, color=RED,  alpha=0.85, label='Fraude', density=True)
axes[0].set_title('Distribution des montants')
axes[0].set_xlabel('Montant (EUR)')
axes[0].set_ylabel('Densite')
axes[0].set_yscale('log')
axes[0].legend()

axes[1].boxplot([normal_amounts, fraud_amounts], labels=['Normal', 'Fraude'], patch_artist=True,
                boxprops=dict(facecolor='#1a1600', color=GOLD), medianprops=dict(color=GREEN, linewidth=2),
                whiskerprops=dict(color=GOLD), capprops=dict(color=GOLD),
                flierprops=dict(marker='o', color=RED, alpha=0.3, markersize=3))
axes[1].set_title('Boxplot des montants')
axes[1].set_ylabel('Montant (EUR)')
plt.tight_layout()
plt.savefig('../outputs/figures/02_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Distribution Temporelle

In [ ]:
df_temp = df.copy()
df_temp['Hour'] = (df_temp['Time'] / 3600) % 24

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Distribution Temporelle', color=GOLD, fontsize=14)

axes[0].hist(df_temp[df_temp['Class']==0]['Hour'], bins=48, color=GOLD, alpha=0.65, label='Normal', density=True)
axes[0].hist(df_temp[df_temp['Class']==1]['Hour'], bins=48, color=RED,  alpha=0.85, label='Fraude', density=True)
axes[0].set_title('Distribution par heure - Normal vs Fraude')
axes[0].set_xlabel('Heure de la journee')
axes[0].set_ylabel('Densite')
axes[0].legend()

df_temp['TimeH'] = df_temp['Time'] / 3600
time_bins = pd.cut(df_temp['TimeH'], bins=48)
fraud_rate = df_temp.groupby(time_bins, observed=False)['Class'].mean() * 100
fraud_rate.index = [i.mid for i in fraud_rate.index]
axes[1].plot(fraud_rate.index, fraud_rate.values, color=RED, linewidth=2)
axes[1].fill_between(fraud_rate.index, fraud_rate.values, alpha=0.3, color=RED)
axes[1].axhline(df['Class'].mean()*100, color=GOLD, linestyle='--', linewidth=1.5, label=f'Taux moyen')
axes[1].set_title('Taux de fraude (%) dans le temps')
axes[1].set_xlabel('Heure (depuis debut du dataset)')
axes[1].set_ylabel('Taux de fraude (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/03_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Correlation des Features avec la Cible

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').sort_values()
top_pos = corr_with_class.tail(5).index.tolist()
top_neg = corr_with_class.head(5).index.tolist()
print(f'Features + correlees : {top_pos}')
print(f'Features - correlees : {top_neg}')

fig, ax = plt.subplots(figsize=(12, 6))
colors = [RED if v < 0 else GREEN for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors, alpha=0.8)
ax.set_title('Correlation des features avec Class (fraude)', color=GOLD)
ax.set_xlabel('Coefficient de correlation')
ax.axvline(x=0, color=GOLD, linewidth=1, linestyle='--')
plt.tight_layout()
plt.savefig('../outputs/figures/04_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Heatmap de Correlation

In [ ]:
important_features = top_neg + top_pos + ['Amount', 'Time', 'Class']
corr_matrix = df[important_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='YlOrBr', mask=mask, ax=ax,
            linewidths=0.8, linecolor='#1a1a1a', annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Heatmap de correlation - Top Features', color=GOLD, fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/05_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Distribution des Features V1-V12

In [ ]:
features = [f'V{i}' for i in range(1, 13)]
normal = df[df['Class'] == 0]
fraud  = df[df['Class'] == 1]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle('Distribution des Features V1-V12 : Normal vs Fraude', color=GOLD, fontsize=14)

for i, (feat, ax) in enumerate(zip(features, axes.flatten())):
    ax.hist(normal[feat], bins=60, density=True, color=GOLD, alpha=0.6, label='Normal')
    ax.hist(fraud[feat],  bins=60, density=True, color=RED,  alpha=0.8, label='Fraude')
    ax.set_title(feat, color=GOLD)
    ax.legend(fontsize=8)
    ax.axvline(normal[feat].mean(), color=GOLD, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(fraud[feat].mean(),  color=RED,  linestyle='--', linewidth=1, alpha=0.7)

plt.tight_layout()
plt.savefig('../outputs/figures/06_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pretraitement des Donnees

In [ ]:
# 3.1 Normalisation de Amount et Time
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled['Amount_scaled'] = scaler.fit_transform(df_scaled[['Amount']])
df_scaled['Time_scaled']   = scaler.fit_transform(df_scaled[['Time']])
df_scaled = df_scaled.drop(columns=['Amount', 'Time'])

print('Normalisation terminee')
print(f'Amount_scaled -> moyenne: {df_scaled["Amount_scaled"].mean():.4f}, std: {df_scaled["Amount_scaled"].std():.4f}')
print(f'Time_scaled   -> moyenne: {df_scaled["Time_scaled"].mean():.4f}, std: {df_scaled["Time_scaled"].std():.4f}')

In [ ]:
# 3.2 Split train/test stratifie 80/20
X = df_scaled.drop(columns=['Class'])
y = df_scaled['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train : {X_train.shape[0]:,} samples | Fraudes: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Test  : {X_test.shape[0]:,} samples  | Fraudes: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

In [ ]:
# 3.3 Reequilibrage avec SMOTE (sur train UNIQUEMENT)
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Avant SMOTE -> {len(y_train):,} | Fraudes: {y_train.sum():,}')
print(f'Apres SMOTE -> {len(y_train_res):,} | Fraudes: {y_train_res.sum():,}')

# Visualisation du reequilibrage
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Avant et Apres SMOTE', color=GOLD, fontsize=13)

axes[0].bar(['Normal', 'Fraude'], [y_train.value_counts()[0], y_train.value_counts()[1]], color=[GOLD, RED], alpha=0.8)
axes[0].set_title('Avant SMOTE', color=GOLD)
axes[0].set_ylabel('Count')

axes[1].bar(['Normal', 'Fraude'], [y_train_res.value_counts()[0], y_train_res.value_counts()[1]], color=[GOLD, RED], alpha=0.8)
axes[1].set_title('Apres SMOTE', color=GOLD)
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Entrainement des Modeles

In [ ]:
import time
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
print(f'Scale pos weight : {scale_pos_weight:.1f}')

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.01, random_state=42, class_weight='balanced'),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'),
    'XGBoost'            : xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM'           : lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, is_unbalance=True, random_state=42, verbose=-1),
}

results = {}

for name, model in models.items():
    print(f'Entrainement : {name}...')
    t0 = time.time()
    model.fit(X_train_res, y_train_res)
    elapsed = time.time() - t0
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_pred_proba': y_pred_proba,
        'roc_auc'  : roc_auc_score(y_test, y_pred_proba),
        'pr_auc'   : average_precision_score(y_test, y_pred_proba),
        'f1'       : f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'train_time': round(elapsed, 2),
    }
    print(f'  AUC-ROC={results[name]["roc_auc"]:.4f} | F1={results[name]["f1"]:.4f} | Temps={elapsed:.1f}s')

# Isolation Forest
print('Entrainement : Isolation Forest...')
iso = IsolationForest(n_estimators=200, contamination=y_train.mean(), random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_scores = -iso.score_samples(X_test)
iso_pred   = (iso.predict(X_test) == -1).astype(int)
results['Isolation Forest'] = {
    'model': iso, 'y_pred': iso_pred, 'y_pred_proba': iso_scores,
    'roc_auc'  : roc_auc_score(y_test, iso_scores),
    'pr_auc'   : average_precision_score(y_test, iso_scores),
    'f1'       : f1_score(y_test, iso_pred),
    'precision': precision_score(y_test, iso_pred, zero_division=0),
    'recall'   : recall_score(y_test, iso_pred, zero_division=0),
    'train_time': 0,
}
print(f'  AUC-ROC={results["Isolation Forest"]["roc_auc"]:.4f}')

### 4.1 Tableau Comparatif

In [ ]:
df_results = pd.DataFrame({
    n: {'AUC-ROC': round(v['roc_auc'],4), 'AUC-PR': round(v['pr_auc'],4),
        'F1': round(v['f1'],4), 'Precision': round(v['precision'],4),
        'Recall': round(v['recall'],4), 'Temps(s)': v['train_time']}
    for n, v in results.items()
}).T.sort_values('AUC-ROC', ascending=False)

print('Tableau comparatif des modeles :')
df_results

## 5. Evaluation des Performances

### 5.1 Courbes ROC

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
COLORS = [GOLD, GREEN, '#60a5fa', '#c084fc', '#fb923c', RED]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Courbes ROC et Precision-Recall', color=GOLD, fontsize=14)

for i, (name, res) in enumerate(results.items()):
    color = COLORS[i % len(COLORS)]
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={res["roc_auc"]:.3f})')
    prec, rec, _ = precision_recall_curve(y_test, res['y_pred_proba'])
    axes[1].plot(rec, prec, color=color, lw=2.5, label=f'{name} (AP={res["pr_auc"]:.3f})')

axes[0].plot([0,1],[0,1],'k--', lw=1.5, alpha=0.5, label='Random')
axes[0].set_title('Courbes ROC', color=GOLD)
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].legend(fontsize=8)

axes[1].axhline(y=y_test.mean(), color='white', linestyle='--', lw=1.5, alpha=0.5, label='Baseline')
axes[1].set_title('Courbes Precision-Recall', color=GOLD)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/10_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Matrices de Confusion

In [ ]:
from sklearn.metrics import confusion_matrix
n = len(results)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Matrices de Confusion', color=GOLD, fontsize=14)
axes_flat = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    tn, fp, fn, tp = cm.ravel()
    annot = np.array([[f'TN\n{tn:,}', f'FP\n{fp:,}'], [f'FN\n{fn:,}', f'TP\n{tp:,}']])
    sns.heatmap(cm, annot=annot, fmt='', ax=axes_flat[i], cmap='YlOrBr',
                linewidths=1.5, linecolor='#1a1a1a',
                xticklabels=['Predit Normal','Predit Fraude'],
                yticklabels=['Reel Normal','Reel Fraude'])
    axes_flat[i].set_title(f'{name}\nAUC={res["roc_auc"]:.3f} | F1={res["f1"]:.3f}', color=GOLD, fontsize=10)

for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/11_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Rapport du Meilleur Modele

In [ ]:
from sklearn.metrics import classification_report
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best_res  = results[best_name]

print(f'Meilleur modele : {best_name}')
print(f'AUC-ROC   : {best_res["roc_auc"]:.4f}')
print(f'AUC-PR    : {best_res["pr_auc"]:.4f}')
print(f'F1-Score  : {best_res["f1"]:.4f}')
print(f'Precision : {best_res["precision"]:.4f}')
print(f'Recall    : {best_res["recall"]:.4f}')
print()
print(classification_report(y_test, best_res['y_pred'], target_names=['Normal', 'Fraude']))

## 6. Feature Importance

In [ ]:
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
    top20 = importances.sort_values(ascending=True).tail(20)
    colors_fi = [RED if v > top20.mean() else GOLD for v in top20.values]

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.suptitle(f'Feature Importance - {best_name} (Top 20)', color=GOLD, fontsize=14)
    ax.barh(top20.index, top20.values, color=colors_fi, alpha=0.85, edgecolor='black')
    ax.axvline(top20.mean(), color=GREEN, linestyle='--', linewidth=1.5, label=f'Moyenne')
    ax.set_xlabel('Importance')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/figures/12_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Top 10 features :')
    print(importances.sort_values(ascending=False).head(10).to_string())

## 7. Optimisation des Hyperparametres (XGBoost)

In [ ]:
# GridSearchCV sur XGBoost
param_grid = {
    'n_estimators'  : [100, 200],
    'max_depth'     : [4, 6],
    'learning_rate' : [0.05, 0.1],
    'subsample'     : [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)

grid_search = GridSearchCV(xgb_base, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train_res, y_train_res)

print(f'Meilleurs hyperparametres : {grid_search.best_params_}')
print(f'Meilleur CV AUC-ROC      : {grid_search.best_score_:.4f}')

best_xgb     = grid_search.best_estimator_
y_pred_opt   = best_xgb.predict(X_test)
y_proba_opt  = best_xgb.predict_proba(X_test)[:, 1]

print(f'AUC-ROC apres optimisation : {roc_auc_score(y_test, y_proba_opt):.4f}')
print(f'F1 apres optimisation      : {f1_score(y_test, y_pred_opt):.4f}')
print()
print(classification_report(y_test, y_pred_opt, target_names=['Normal', 'Fraude']))

## 8. MLflow Tracking

In [ ]:
mlflow.set_experiment('fraud_detection')

for name, res in results.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_param('model_type', name)
        mlflow.log_param('resampling', 'SMOTE')
        mlflow.log_param('test_size', 0.2)
        mlflow.log_metric('roc_auc',   res['roc_auc'])
        mlflow.log_metric('pr_auc',    res['pr_auc'])
        mlflow.log_metric('f1_score',  res['f1'])
        mlflow.log_metric('precision', res['precision'])
        mlflow.log_metric('recall',    res['recall'])
        try:
            mlflow.sklearn.log_model(res['model'], name.replace(' ', '_').lower())
        except:
            pass
        print(f'Run enregistre : {name}')

print('Pour visualiser les runs : mlflow ui')

## 9. Sauvegarde du Meilleur Modele

In [ ]:
best_model = results[best_name]['model']

joblib.dump(best_model, '../models/best_fraud_model.pkl')
joblib.dump(scaler,     '../models/scaler.pkl')

print(f'Modele sauvegarde : ../models/best_fraud_model.pkl')
print(f'Scaler sauvegarde : ../models/scaler.pkl')
print(f'Meilleur modele   : {best_name}')
print(f'AUC-ROC           : {results[best_name]["roc_auc"]:.4f}')

## 10. Prediction en Temps Reel

In [ ]:
def predict_new_transaction(transaction, model, scaler, threshold=0.5):
    df_tx = pd.DataFrame([transaction])
    df_tx['Amount_scaled'] = scaler.transform(df_tx[['Amount']])
    df_tx['Time_scaled']   = scaler.transform(df_tx[['Time']])
    df_tx = df_tx.drop(columns=['Amount', 'Time'])
    proba = float(model.predict_proba(df_tx)[0][1])
    is_fraud = proba >= threshold
    if proba < 0.3:   risk = 'LOW'
    elif proba < 0.5: risk = 'MEDIUM'
    elif proba < 0.75: risk = 'HIGH'
    else: risk = 'CRITICAL'
    return {'is_fraud': is_fraud, 'probability': round(proba, 4), 'risk': risk}

# Transaction normale
tx_normal = {
    'Time': 80000,
    'V1':0.23,'V2':0.05,'V3':0.22,'V4':0.21,'V5':-0.01,'V6':0.12,'V7':0.08,'V8':0.03,
    'V9':-0.02,'V10':0.01,'V11':0.04,'V12':-0.03,'V13':-0.01,'V14':0.02,'V15':0.01,'V16':-0.01,
    'V17':-0.01,'V18':0.00,'V19':0.01,'V20':0.00,'V21':0.00,'V22':0.01,'V23':0.00,'V24':0.00,
    'V25':0.01,'V26':0.00,'V27':0.00,'V28':0.00,'Amount': 45.50
}

# Transaction frauduleuse
tx_fraud = {
    'Time': 80000,
    'V1':-3.04,'V2':2.11,'V3':-3.58,'V4':3.25,'V5':-2.88,'V6':-1.59,'V7':-2.62,'V8':0.84,
    'V9':-1.57,'V10':-4.46,'V11':3.15,'V12':-7.48,'V13':0.13,'V14':-6.45,'V15':0.27,'V16':-2.75,
    'V17':-4.61,'V18':-1.25,'V19':-1.09,'V20':-0.37,'V21':-0.54,'V22':-0.20,'V23':-0.24,'V24':0.11,
    'V25':-0.23,'V26':0.30,'V27':-0.01,'V28':0.01,'Amount': 2450.00
}

r1 = predict_new_transaction(tx_normal, best_model, scaler)
r2 = predict_new_transaction(tx_fraud,  best_model, scaler)

print('Transaction normale (45.50 EUR) :')
print(f'  Fraude: {r1["is_fraud"]} | Probabilite: {r1["probability"]} | Risque: {r1["risk"]}')
print()
print('Transaction suspecte (2450.00 EUR) :')
print(f'  Fraude: {r2["is_fraud"]} | Probabilite: {r2["probability"]} | Risque: {r2["risk"]}')

## 11. Conclusion

### Resultats obtenus
- **Meilleur modele** : XGBoost (AUC-ROC > 0.98)
- **Features les plus importantes** : V14, V10, V4, V12, V17
- **SMOTE** a permis de corriger le desequilibre de classes (0.17% de fraudes)
- **Faux negatifs** minimises pour eviter les fraudes non detectees

### Points cles
1. Le dataset est fortement desequilibre (1 fraude pour 577 transactions normales)
2. L'AUC-PR est plus pertinente que l'AUC-ROC pour les classes desequilibrees
3. Abaisser le seuil de decision permet d'augmenter le recall (moins de fraudes manquees)
4. Les features V14 et V10 sont les plus discriminantes selon XGBoost

### Pistes d'amelioration
- Autoencoders pour la detection d'anomalies (Deep Learning)
- SHAP pour l'interpretabilite du modele
- Monitoring en production avec MLflow
- API de prediction avec FastAPI